# 00 — Preparar el dataset

Este notebook descarga productos de `Shopify/product-catalogue`
(Hugging Face), los mapea a las 21 categorías de NovaStore, los traduce
a español cuando hace falta, y guarda `train.csv`/`val.csv`/`test.csv` en
tu Google Drive — **todos los demás notebooks leen esa misma partición**,
así que corre este primero.

Tiempo esperado en Colab (GPU T4 gratuita, pero acá casi no se usa GPU —
es sobre todo descarga y procesamiento de imágenes): entre 20 y 45
minutos para el dataset completo (300 ejemplos/clase), según qué tan
rápido esté Hugging Face ese día.

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {{CARPETA_ML}}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): usa la carpeta ml/ del repo.
    CARPETA_ML = str(__import__("pathlib").Path.cwd())

sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
from pathlib import Path

MODO_RAPIDO = False  # True = dataset diminuto (3000 filas revisadas, 5 ejemplos/clase) para probar que todo corre
CARPETA_DATOS = Path(CARPETA_ML) / "datos"
MAX_POR_CLASE = 5 if MODO_RAPIDO else 300
LIMITE_FILAS = 3000 if MODO_RAPIDO else None

print(f"MODO_RAPIDO={MODO_RAPIDO} · max_por_clase={MAX_POR_CLASE} · limite_filas={LIMITE_FILAS}")


## 1. Descargar, filtrar y mapear

`construir_dataset` recorre el dataset en streaming (no descarga los 9.5
GB de una sola vez), descarta lo que no tiene imagen o título, mapea la
categoría con `mapeo_categorias.py`, y corta cada clase en
`MAX_POR_CLASE` ejemplos. Al final imprime cuántos ejemplos quedaron por
categoría — si alguna aparece "DÉBIL", revisa `ml/DATASET.md`.

In [ ]:
from preparar_dataset import construir_dataset

df = construir_dataset(
    max_por_clase=MAX_POR_CLASE,
    limite_filas=LIMITE_FILAS,
    carpeta_salida=CARPETA_DATOS,
)
df.head()


## 2. Traducir a español

Los textos que `langdetect` no identifica como español se traducen con
`Helsinki-NLP/opus-mt-en-es`. Ver la limitación metodológica (textos en
un tercer idioma) en `ml/DATASET.md`.

In [ ]:
from preparar_dataset import traducir_a_espanol

df = traducir_a_espanol(df)
df[["titulo", "texto_es"]].sample(min(5, len(df)))


## 3. Partir en train/val/test (70/15/15, semilla 42) y guardar

In [ ]:
from preparar_dataset import guardar_particion, particionar

train, val, test = particionar(df, estratificar=not MODO_RAPIDO)
guardar_particion(train, val, test, CARPETA_DATOS)


## 4. Revisar: conteo por clase y algunos ejemplos con imagen

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

print(train["categoria"].value_counts())

muestra = train.sample(min(8, len(train)), random_state=42)
fig, ejes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, fila) in zip(ejes.flat, muestra.iterrows()):
    try:
        ax.imshow(Image.open(fila["ruta_imagen"]))
    except Exception:
        pass
    ax.set_title(fila["categoria"], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
